In [ ]:
#归一化 height 列
df['height'] = df['height'] / df['height'].max()
#这行代码的作用是同时查看 length、width、height 三列归一化后的前 5 行数据，用来验证归一化是否成功。
df[['length','width','height']].head()

#这行代码的作用是把 horsepower 列从 float 类型转换为 int（整数）类型。
# 用 copy=True 更安全，避免意外修改其他引用同一数据的地方。不过在新版 Pandas 中这个参数行为有所变化，这里可以理解为确保生成一份全新的 int 数据。
df["horsepower"]=df["horsepower"].astype(int, copy=True)

#开始用matplotlib画图
import matplotlib as plt
from matplotlib import pyplot

#通常标准写法是 import matplotlib.pyplot as plt
#绘制直方图（自动将数据分成若干个区间，统计每个区间的频次），10-20，20-30，30-40等等划分区间
plt.pyplot.hist(df["horsepower"])

# 设置图表的x轴和y轴以及图表的标题，x轴是各种马力，y轴是各马力统计数
plt.pyplot.xlabel("horsepower")
plt.pyplot.ylabel("count")
plt.pyplot.title("horsepower bins")

#这行代码的作用是创建 3 个等宽分箱的边界点。np.linspace(start, stop, num) 是什么？NumPy 的 linspace 函数：在 start 和 stop 之间等间距生成 num 个点。
#生成 4 个点（= 3 个区间需要 4 个边界）
bins = np.linspace(min(df["horsepower"]), max(df["horsepower"]), 4)
#直接输出结果 bins = [48.0, 128.0, 208.0, 288.0]
bins
#将三个区间取名字
group_names = ['Low', 'Medium', 'High']

#bins分箱边界（上一行用 np.linspace 创建的）;labels=group_names(每个区间的标签名);include_lowest=True包含最小值，即 48 这个值也归入第一个区间（否则默认左开右闭，48 会被排除）
#pd.cut 默认是左开右闭区间 (a, b]，加上 include_lowest=True 后第一个区间变为左闭右闭 [a, b]
# 区间1: [48,  128]  → "Low"      ← 包含 48（因为 include_lowest）
# 区间2: (128, 208]  → "Medium"
# 区间3: (208, 288]  → "High"
#新增一列horsepower-binned，里面的值是Low、Medium、High
df['horsepower-binned'] = pd.cut(df['horsepower'], bins, labels=group_names, include_lowest=True )
#显示原马力值和分箱后的值，对比验证
df[['horsepower','horsepower-binned']].head(20)
#统计每个分箱中有多少辆车。
df["horsepower-binned"].value_counts()


#直方图回答"数据长什么样？"，柱状图回答"每类有多少？"
#用 hist() 的场景：想看原始数据的分布，初步想知道马力数据整体长什么样？集中在哪？；用 bar() 的场景：已经有了分类统计结果，数据已经手动分好算好了
#在本实验中，先用直方图看马力分布 → 然后分箱 → 最后用柱状图展示分箱结果。是一个从连续到离散的过程
import matplotlib as plt
from matplotlib import pyplot
#group_names表示x轴上的3个柱子标签，df["horsepower-binned"].value_counts()每根柱子的高度（各箱车辆数）
pyplot.bar(group_names, df["horsepower-binned"].value_counts())
plt.pyplot.xlabel("horsepower")
plt.pyplot.ylabel("count")
plt.pyplot.title("horsepower bins")


import matplotlib as plt
from matplotlib import pyplot
# bins = 3 这是用 hist() 手动指定 3 个分箱来画直方图，效果上等价于前面用 pd.cut() 分箱后再用 bar() 画柱状图的结果。
plt.pyplot.hist(df["horsepower"], bins = 3)
plt.pyplot.xlabel("horsepower")
plt.pyplot.ylabel("count")
plt.pyplot.title("horsepower bins")

#列出 DataFrame 中所有列的名称。
df.columns

#这是创建指示变量（Indicator Variable / Dummy Variable），把文字类别转成数字 0/1。
#fuel-type 列目前存的是文字 pd.get_dummies() 会自动扫描所有唯一值，为每个值创建一列，用 0/1 标记
#pd.get_dummies() 会根据 fuel-type 列中有多少种不同的值，创建对应数量的新列。
# 注意！！！这行代码做的事情是：
# 从 df 中取出 fuel-type 这一列
# 基于它创建了一个全新的、独立的 DataFrame（叫 dummy_variable_1）
# 原来的 df 完全没有变化
#    diesel  gas
# 0       0    1
# 1       1    0
# 2       0    1
# 3       0    1
# 4       1    0
#解析规则
# gas	diesel=0, gas=1
# diesel	diesel=1, gas=0
#为什么要这样做？因为回归模型不认识文字，只认识数字。 “gas”字符串无法进行计算
dummy_variable_1 = pd.get_dummies(df["fuel-type"])
dummy_variable_1.head()
#改名字
dummy_variable_1.rename(columns={'gas':'fuel-type-gas', 'diesel':'fuel-type-diesel'}, inplace=True)
dummy_variable_1.head()

#拼接合并，把原表 df 和 dummy 表左右拼接，之前df表只有26列，现在有28列，形成一张完整的表，axis=1，新增的是列
df = pd.concat([df, dummy_variable_1], axis=1)
# 将fuel-type这一列删除，axis=1，按列删除，并覆盖原表
df.drop("fuel-type", axis = 1, inplace=True)
df.head()
